# Wizualizacja Uprawnień Looker - Wykres Sankey

Poniższy kod ładuje plik `sankey_data.json` i generuje piękny, interaktywny wykres przepływowy za pomocą biblioteki `Plotly`. 
Wykres prezentuje ścieżkę:
`Model -> Explore -> Dashboard -> Group -> User`

*Aby wykres działał, upewnij się że uruchomiłeś skrypty `extract_raw_data.py` oraz `build_sankey_data.py`.*

In [ ]:
# Opcjonalnie: Przebuduj plik sankey_data.json bez wychodzenia z Notatnika!
from build_sankey_data import build_sankey

# Uruchom tę komórkę, aby zmienić parametry wykresu w locie:
build_sankey(
    input_file="raw_looker_data.json",
    output_file="sankey_data.json",
    include_users=False,        # Zmień na True, by włączyć użytkowników na końcu wykresu
    target_models=None          # Zmień na np. ["model_1", "model_2"], by filtrować konkretne modele
)

In [ ]:
import json
import plotly.graph_objects as go

# 1. Wczytanie przygotowanych danych Sankeya
try:
    with open('sankey_data.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError("Plik sankey_data.json nie istnieje! Odpal skrypt build_sankey_data.py.")

nodes = data.get('nodes', [])
links = data.get('links', [])

# 2. Przypisanie ładnych kolorów dla węzłów w zależności od typu
type_colors = {
    "model": "rgba(255, 77, 77, 0.8)",
    "explore": "rgba(255, 166, 77, 0.8)",
    "dashboard": "rgba(77, 77, 255, 0.8)",
    "group": "rgba(255, 77, 255, 0.8)",
    "user": "rgba(139, 69, 19, 0.8)"
}

node_labels = []
node_colors = []

for node in nodes:
    node_labels.append(node.get('label', 'Unknown'))
    node_colors.append(type_colors.get(node.get('type'), "rgba(200, 200, 200, 0.8)"))

# 3. Przygotowanie wektorów źródeł i celów dla Plotly
sources = []
targets = []
values = []

for link in links:
    sources.append(link['source'])
    targets.append(link['target'])
    values.append(link.get('value', 1))

if not nodes or not links:
    print("Brak danych do wyrysowania wykresu. Upewnij się, że wyeksportowałeś jakieś modele.")
else:
    # 4. Konstrukcja i Rysowanie Wykresu Sankey
    fig = go.Figure(data=[go.Sankey(
        node = dict(
            pad = 20,
            thickness = 30,
            line = dict(color = "black", width = 0.5),
            label = node_labels,
            color = node_colors
        ),
        link = dict(
            source = sources,
            target = targets,
            value = values,
            color = "rgba(200, 200, 200, 0.3)" # Lekko przezroczyste szare strumienie
        )
    )])

    fig.update_layout(
        title_text="Przepływ Uprawnień: Od Modelu do Użytkownika",
        font_size=12,
        height=800,
        margin=dict(l=20, r=20, t=60, b=20)
    )

    fig.show()
